# Notebook 2 — Feature Engineering (Custom Transformer + TF-IDF)

**Module:** 7006SCN Machine Learning and Big Data — Coventry University  
**Project:** Scalable Fake News Detection Using Distributed ML on Common Crawl  

---

## Pipeline stages
```
raw text → TextStatisticsTransformer → Tokenizer → StopWordsRemover → HashingTF → IDF → VectorAssembler → feature vector
```

1. **Custom `TextStatisticsTransformer`** — domain-specific feature extractor that captures stylistic cues (text length, capitalisation ratio, exclamation count) directly from raw text *before* cleaning. Runs entirely via Spark SQL functions (no Python UDFs).
2. **TF-IDF pipeline** — standard bag-of-words with 2^16 hash buckets.
3. **VectorAssembler** — combines the 65 536-dim TF-IDF vector with the 5 text-statistic features → 65 541-dim final vector.

All transformations are **Spark MLlib Transformers** — they run distributed
across executors, not on the driver.

In [ ]:
# ── 2.1  Bootstrap ──────────────────────────────────────────────────────
import os, sys

os.environ["JAVA_HOME"]             = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.16.8-hotspot"
os.environ["HADOOP_HOME"]           = r"C:\hadoop"
os.environ["PATH"]                  = r"C:\hadoop\bin;" + os.environ.get("PATH", "")
os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession, functions as F
from pyspark import StorageLevel
from pyspark.ml import Transformer

spark = (
    SparkSession.builder
    .appName("FakeNewsDetection")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} | UI: {spark.sparkContext.uiWebUrl}")

In [ ]:
# ── 2.2  Load Parquet produced by Notebook 1 ────────────────────────────
PARQUET_PATH = "../data/parquet/news_articles"

df = spark.read.parquet(PARQUET_PATH)
print(f"Loaded {df.count():,} rows  |  Partitions: {df.rdd.getNumPartitions()}")
df.printSchema()
df.groupBy("label").count().show()

## 2.3  Label Strategy

The dataset generated in Notebook 1 already contains labels:
- **label = 0** → Reliable news articles  
- **label = 1** → Fake / unreliable news articles  

In a production Common Crawl pipeline, labels would come from one of:
- **Option A — Domain-based weak labelling:** Map URLs to curated lists of reliable vs. unreliable domains (MediaBiasFactCheck, OpenSources).
- **Option B — Supervised join:** Cross-reference URLs against labelled datasets like FakeNewsNet, LIAR, or ISOT.

Since our data is pre-labelled, we proceed directly to text preprocessing.

In [ ]:
# ── 2.3b  Custom Transformer — Domain-Specific Text Statistics ───────────
# This transformer extracts stylistic features from RAW text (before cleaning)
# that help distinguish fake news from reliable news:
#   - text_length        : total characters (fake articles tend to be shorter)
#   - word_count         : total words
#   - avg_word_length    : mean characters per word
#   - caps_ratio         : proportion of uppercase letters (fake uses MORE CAPS)
#   - exclamation_count  : number of '!' (sensationalism indicator)
#
# References:
#   Horne & Adali (2017) — "This Just In: Fake News Packs a Lot in Title"
#   Pérez-Rosas et al. (2018) — surface-level stylistic cues complement BOW features
#
# IMPLEMENTATION NOTE: Uses ONLY Spark SQL built-in functions (F.length, F.size,
# F.regexp_replace, etc.) — NO Python UDFs.  This is critical on Windows where
# PySpark 4.x Arrow-only mode may crash Python worker processes.

class TextStatisticsTransformer(Transformer):
    """Custom domain-specific Transformer for fake news text analysis."""

    def __init__(self, inputCol="text"):
        super().__init__()
        self._inputCol = inputCol

    def _transform(self, dataset):
        ic = self._inputCol
        return (
            dataset
            .withColumn("text_length",
                        F.length(F.col(ic)).cast("double"))
            .withColumn("word_count",
                        F.size(F.split(F.col(ic), r"\s+")).cast("double"))
            .withColumn("avg_word_length",
                        F.when(F.col("word_count") > 0,
                               F.col("text_length") / F.col("word_count"))
                         .otherwise(0.0))
            .withColumn("caps_ratio",
                        F.when(F.col("text_length") > 0,
                               F.length(F.regexp_replace(F.col(ic), r"[^A-Z]", ""))
                                .cast("double") / F.col("text_length"))
                         .otherwise(0.0))
            .withColumn("exclamation_count",
                        (F.length(F.col(ic))
                         - F.length(F.regexp_replace(F.col(ic), "!", "")))
                        .cast("double"))
        )

# Apply custom transformer on RAW text (BEFORE cleaning strips capitalisation & punctuation)
text_stats_transformer = TextStatisticsTransformer(inputCol="text")
df = text_stats_transformer.transform(df)
print("Custom TextStatisticsTransformer applied — added 5 text-statistic columns")
df.select("text_length", "word_count", "avg_word_length", "caps_ratio", "exclamation_count").describe().show()

# Export per-article text stats for Tableau (useful for EDA visualisation)
import pandas as pd
text_stats_pdf = df.select(
    "label", "text_length", "word_count", "avg_word_length", "caps_ratio", "exclamation_count"
).toPandas()
text_stats_pdf["label_name"] = text_stats_pdf["label"].map({0: "Reliable", 1: "Unreliable"})
text_stats_pdf.to_csv("../tableau/text_statistics.csv", index=False)
print("✓ Exported text_statistics.csv for Tableau EDA dashboard")

In [ ]:
# ── 2.4  Text preprocessing (no UDFs — all Spark built-in functions) ─────
from pyspark.sql.functions import regexp_replace, trim, length, lower, col

labelled_df = (
    df
    # Remove URLs embedded in text
    .withColumn("text", regexp_replace("text", r"https?://\S+", ""))
    # Remove non-alphabetic characters (keep spaces)
    .withColumn("text", regexp_replace("text", r"[^a-zA-Z\s]", ""))
    # Collapse multiple spaces
    .withColumn("text", regexp_replace("text", r"\s+", " "))
    # Trim & lowercase
    .withColumn("text", trim(lower(col("text"))))
    # Drop if text too short after cleaning
    .filter(length("text") >= 100)
)

print(f"After text cleaning: {labelled_df.count():,} rows")
labelled_df.select("text").show(2, truncate=120)

In [ ]:
# ── 2.5  Verify text preprocessing ──────────────────────────────────────
print("Label distribution after cleaning:")
labelled_df.groupBy("label").count().show()
print(f"Sample cleaned text (first 200 chars):")
labelled_df.select("text").show(3, truncate=200)

In [ ]:
# ── 2.6  MLlib TF-IDF Pipeline + VectorAssembler ────────────────────────
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF, VectorAssembler
from pyspark.ml import Pipeline

# Stage 1: Tokenizer — splits text into word array
tokenizer = Tokenizer(inputCol="text", outputCol="words")

# Stage 2: StopWordsRemover — removes English stop words
stopwords_remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")

# Stage 3: HashingTF — maps words to fixed-size feature vector
#   numFeatures=2**16 (65536) — good balance: low collision rate, fits in memory
NUM_FEATURES = 2**16
hashing_tf = HashingTF(inputCol="filtered_words", outputCol="raw_features", numFeatures=NUM_FEATURES)

# Stage 4: IDF — down-weight common terms, up-weight rare terms
#   Output is "tfidf_features" (not "features") because we combine with text_stats next
idf = IDF(inputCol="raw_features", outputCol="tfidf_features", minDocFreq=5)

feature_pipeline = Pipeline(stages=[tokenizer, stopwords_remover, hashing_tf, idf])

print("TF-IDF pipeline stages:")
for i, stage in enumerate(feature_pipeline.getStages()):
    print(f"  {i}: {type(stage).__name__}")
print(f"\nHashingTF numFeatures = {NUM_FEATURES:,} (2^16)")
print(f"Text statistics features = 5  (from Custom Transformer)")
print(f"TOTAL feature dimensions = {NUM_FEATURES + 5:,}")

In [ ]:
# ── 2.7  Fit TF-IDF pipeline + Combine with custom text statistics ───────
labelled_df.persist(StorageLevel.MEMORY_AND_DISK)

pipeline_model = feature_pipeline.fit(labelled_df)
features_df = pipeline_model.transform(labelled_df)

# Combine TF-IDF (65 536-dim) + text_stats (5-dim) → final "features" (65 541-dim)
TEXT_STATS_COLS = ["text_length", "word_count", "avg_word_length", "caps_ratio", "exclamation_count"]
text_stats_assembler = VectorAssembler(
    inputCols=TEXT_STATS_COLS, outputCol="text_stats", handleInvalid="skip"
)
feature_combiner = VectorAssembler(
    inputCols=["tfidf_features", "text_stats"], outputCol="features", handleInvalid="skip"
)
features_df = text_stats_assembler.transform(features_df)
features_df = feature_combiner.transform(features_df)
print(f"Combined features: TF-IDF ({NUM_FEATURES}) + text_stats ({len(TEXT_STATS_COLS)}) = {NUM_FEATURES + len(TEXT_STATS_COLS)} dimensions")

# Select only columns needed downstream
features_df = features_df.select("title", "subject", "date", "source", "label", "features")

features_df.persist(StorageLevel.MEMORY_AND_DISK)
print(f"Feature vectors: {features_df.count():,} rows")
features_df.show(3, truncate=60)

labelled_df.unpersist()

In [ ]:
# ── 2.8  Train / Validation / Test Split with Stratification ────────────
# Stratified split: maintain label ratio across all splits
# Spark doesn't have sklearn's StratifiedKFold, so we use sampleBy

fractions = {0: 0.7, 1: 0.7}   # 70% of each class for train
train_df = features_df.stat.sampleBy("label", fractions, seed=42)
remaining = features_df.subtract(train_df)

fractions_vt = {0: 0.5, 1: 0.5}  # 50/50 split of remaining → ~15% val, ~15% test
val_df  = remaining.stat.sampleBy("label", fractions_vt, seed=42)
test_df = remaining.subtract(val_df)

for name, split_df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    total = split_df.count()
    pos   = split_df.filter(col("label") == 1).count()
    print(f"{name:6s}: {total:>8,} rows  |  label=1 ratio: {pos/max(total,1):.3f}")

In [ ]:
# ── 2.9  Save feature-engineered splits to Parquet ──────────────────────
FEATURES_BASE = "../data/parquet/features"

for name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    path = f"{FEATURES_BASE}/{name}"
    split_df.write.mode("overwrite").parquet(path)
    print(f"✓ {name} → {path}")

# Save fitted pipeline model for reproducibility
pipeline_model.write().overwrite().save("../data/models/feature_pipeline")
print("✓ Feature pipeline model saved")

In [ ]:
# ── 2.10  Class distribution + Imbalance Analysis ───────────────────────
import pandas as pd

class_dist = (
    features_df
    .groupBy("label")
    .agg(F.count("*").alias("count"))
    .toPandas()
)
class_dist["label_name"] = class_dist["label"].map({0: "Reliable", 1: "Unreliable"})
class_dist.to_csv("../tableau/class_distribution.csv", index=False)
print("✓ Exported class_distribution.csv for Tableau\n")

# ── CLASS IMBALANCE ANALYSIS ──
total_rows = class_dist["count"].sum()
print("=== CLASS IMBALANCE ANALYSIS ===")
for _, row in class_dist.iterrows():
    pct = row["count"] / total_rows * 100
    print(f"  {row['label_name']:12s}: {row['count']:>8,} ({pct:.1f}%)")

imbalance_ratio = class_dist["count"].max() / class_dist["count"].min()
print(f"  Imbalance ratio: {imbalance_ratio:.2f}")
if imbalance_ratio < 1.5:
    print("  → Nearly balanced. Stratified splitting preserves label ratios.")
    print("  → No resampling / class weighting required.")
else:
    print("  → Imbalanced! Consider class weights, SMOTE, or under-sampling.")

class_dist

In [ ]:
# ── Cleanup ──
features_df.unpersist()
# spark.stop()